[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-08a-a8-kickoff.ipynb)

# Assignment 8 kickoff: from your cleaned corpus to a running model

**This notebook is the executable path through Assignment 8.** Every threshold below is quoted
from the handout; if the two ever disagree, the handout governs and this notebook is wrong.

What "done for today" means: your data is split, verified and saved to Drive, and a training
run is going with checkpoints landing in Drive. The run continues after class. You do not wait
for it.

**Running in Google Colab? Two things, in this order:**

1. **Runtime → Change runtime type → GPU, and choose an A100, L4 or G4, not a T4.** Do this
   *first*: changing it later restarts the session and throws away everything you have run. A
   T4 has too little memory for this model with a large word vocabulary, and the setup cell
   refuses to continue without a GPU, because on a CPU this would take days.
2. **Run the next two cells.** The first installs TorchLingo; the second checks the GPU and
   connects Google Drive, where everything this notebook writes will live.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 8a (*Neural MT: Encoder-Decoder, and Why Attention Was Invented*). A head start on **A8**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
# Check for a GPU and connect Google Drive. Stops with instructions if there is no GPU.
from torchlingo.colab import setup

env = setup(gpu=True, drive=True)
device = env.device
DRIVE = env.drive_dir      # MyDrive in Colab; this folder anywhere else

import torch
from torchlingo.config import Config
from torchlingo.data_processing import create_dataloaders
from torchlingo.diagnostics import check_contamination, padding_report, uniform_loss
from torchlingo.inference import translate_batch
from torchlingo.models import SimpleTransformer
from torchlingo.preprocessing import parallel_txt_to_dataframe, split_exact
from torchlingo.training import train_model
from torchlingo.training_checkpoint import TrainingCheckpointer

## The numbers, and where they come from

The Assignment 8 handout's thresholds, in one place. **If the handout changes, change this
cell; nothing else in the notebook holds a number.**

In [ ]:
# From the Assignment 8 handout ("What To Do"). The handout governs; this cell follows it.
TRAIN_FLOOR = 100_000   # at least this many training pairs, or all of it if you have less
VAL_SIZE    = 2_000
TEST_SIZE   = 2_000
MAX_TOKENS  = 100       # drop a pair if either side is longer than this; a token is a word, or a
                        # character on a target side written without spaces (Step 0 says which)
EPOCHS      = 35        # about 35; if val loss is still falling at the end, raise it and re-run Step 7

SEED = 479              # so a re-run gives you the same split, and the same test set

## Step 0: point at your Assignment 5 corpus

Edit the two paths. They are the two sentence-aligned files you submitted for Assignment 5:
one line per segment, line *n* of one file translating line *n* of the other. Lecture 7 had you
put them in `MyDrive/CS479`.

**Is your language written without spaces between words?** Chinese, Japanese, Thai, Lao, Khmer
and Burmese are. Then set `NO_SPACES = True`. Everything below splits sentences into words at
the spaces, and on your text that makes each whole sentence one "word": almost none repeat, the
vocabulary is nearly all `<unk>`, and the model learns to write `<unk>` and nothing else. With
the switch on, the notebook does three things differently: it caps length in characters (Step 1),
it builds a **subword** vocabulary (Step 6), and it scores with a BLEU that does not need spaces.
Subwords are what Lecture 9 and Assignment 9 are about; you are starting them a week early,
because without them your language cannot be translated at all.

In [ ]:
# TODO: your two Assignment 5 files. English on the source side; your language on the target.
SRC_FILE = DRIVE / "CS479" / "cleaned.en.txt"       # <- edit
TGT_FILE = DRIVE / "CS479" / "cleaned.xx.txt"       # <- edit

# True if your language is written without spaces between words (Chinese, Japanese, Thai, ...).
NO_SPACES = False                                   # <- edit if so

# Everything this notebook writes goes here, so you can find it and so a dropped session loses
# nothing.
OUT_DIR = DRIVE / "CS479" / "assignment8"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Raises if the two files have different line counts: then they are not sentence-aligned, and
# Assignment 5 needs fixing before anything below means something.
pairs = parallel_txt_to_dataframe(SRC_FILE, TGT_FILE)
pairs = pairs[(pairs.src != "") & (pairs.tgt != "")].reset_index(drop=True)
print(f"Loaded {len(pairs):,} pairs.")
if len(pairs) < TRAIN_FLOOR + VAL_SIZE + TEST_SIZE:
    print(f"That is below {TRAIN_FLOOR + VAL_SIZE + TEST_SIZE:,}, so the handout's low-resource "
          "clause applies to you: use all of it, and say so in your write-up.")

# A check on the switch: on text without spaces, a "sentence" averages one or two "words".
words_per_line = pairs.tgt.str.split().str.len().mean()
if words_per_line < 3 and not NO_SPACES:
    print(f"\nYour target sentences average {words_per_line:.1f} words. That looks like a language "
          "written without spaces: read the note above Step 0 and set NO_SPACES = True.")

## Step 1: cap sentence length, *before* splitting

The cap removes pairs. If you split first and cap afterwards, a training set of exactly
100,000 pairs quietly becomes fewer than 100,000, and you no longer meet the floor you think you
meet. So the cap goes first, and the split sizes you see below are the sizes you submit.

Why cap at all: peak memory in training is set by the *longest* sentence in a batch, not the
typical one, and the few very long pairs cost the most. The cell prints how many of yours it
drops.

In [ ]:
src_len = pairs.src.str.split().str.len()
if NO_SPACES:
    # No spaces to count words by, so count characters: a character is never more than one
    # subword piece, so this caps the target side in Step 6's tokens too.
    tgt_len = pairs.tgt.str.replace(" ", "").str.len()
else:
    tgt_len = pairs.tgt.str.split().str.len()

keep = (src_len <= MAX_TOKENS) & (tgt_len <= MAX_TOKENS)
dropped = int((~keep).sum())
print(f"Longest source: {src_len.max()} tokens. Longest target: {tgt_len.max()} tokens.")
print(f"Cap at {MAX_TOKENS}: dropping {dropped:,} pairs ({100*dropped/len(pairs):.2f}%).")

pairs = pairs[keep].reset_index(drop=True)
print(f"{len(pairs):,} pairs remain.")

## Step 2: deduplicate on the source side

Not on the pair: on the source. Two identical source sentences with different translations are
still the same source, and if one lands in training and the other in test, your test set
contains sentences the model was trained on.

In [ ]:
before = len(pairs)
pairs = pairs.drop_duplicates(subset="src", keep="first").reset_index(drop=True)
removed = before - len(pairs)
print(f"Removed {removed:,} pairs with a repeated source ({100*removed/before:.2f}%).")
print(f"{len(pairs):,} unique-source pairs remain.")

## Step 3: split, with exact counts, seeded

The handout asks for 2,000 validation and 2,000 test pairs and everything else in training.
`split_exact` takes those counts. It is seeded: run this cell twice and you get the same test
set both times, which is the only way your BLEU tomorrow is comparable with your BLEU today.

In [ ]:
train, val, test = split_exact(pairs, n_val=VAL_SIZE, n_test=TEST_SIZE, seed=SEED)

print(f"train {len(train):,}   val {len(val):,}   test {len(test):,}")
if len(train) < TRAIN_FLOOR:
    print(f"Training set is below the {TRAIN_FLOOR:,} floor. That is allowed for low-resource "
          "languages: use all of it, and state the count in your write-up.")

## Step 4: verify the split

The check raises rather than prints: a warning you can scroll past is not a check. A
contaminated split here is carried into Assignments 9, 13 and 14.

In [ ]:
for name, held_out in (("test", test), ("val", val)):
    result = check_contamination(held_out.src, train.src, name=name)
    print(f"{name}: {result.detail}")
    if not result:
        raise SystemExit(
            f"The {name} set shares sources with training. Re-run from Step 2 in order; if it "
            "still fails, ask the teaching staff before training anything."
        )

overlap = set(test.src) & set(val.src)
if overlap:
    raise SystemExit(f"{len(overlap)} sources appear in both val and test. Re-run from Step 3.")
print("Clean. Three disjoint sets.")

## Step 5: write the files to Drive

Two formats for two readers: TorchLingo reads one TSV per split, and the handout asks you to
submit the splits as parallel text, six files. Both are written here.

In [ ]:
for name, frame in (("train", train), ("val", val), ("test", test)):
    frame.to_csv(OUT_DIR / f"{name}.tsv", sep="\t", index=False)
    (OUT_DIR / f"{name}.src").write_text("\n".join(frame.src) + "\n", encoding="utf-8")
    (OUT_DIR / f"{name}.tgt").write_text("\n".join(frame.tgt) + "\n", encoding="utf-8")

for p in sorted(OUT_DIR.iterdir()):
    print(f"{p.name:12} {p.stat().st_size/1e6:7.2f} MB")

## Step 6: the model

The course configuration: `d_model` 512, 8 heads, 6 encoder and 6 decoder layers, feed-forward
2048: the architecture of the original Transformer paper. With an 8,000-piece subword vocabulary
it has 56.4 million parameters; your word vocabulary is larger, so your embedding tables are too,
and the cell prints your exact count.

This size translated better than a model a fifth its size and converged in about 30 epochs where
the smaller one needed 65; the trainer keeps your best checkpoint by validation loss, so running
longer costs time, not quality.

**Training batches are bucketed by length** (`use_bucketing=True`). A batch is padded to its
longest sentence, so bucketing groups sentences of similar length; on a 100,000-pair corpus it
removed about three quarters of the padded tokens, and `padding_report` prints yours.

`num_workers=0` is set explicitly. The library default is 4, and on Colab that costs about
twenty seconds of fixed overhead per loader for no gain at any corpus size we measured.

**With `NO_SPACES = True`**, this cell first trains a SentencePiece vocabulary of 8,000 pieces on
`train.tsv` only (never on test: a vocabulary is part of what a model learns), shared by English
and your language. It takes a minute or two, once: re-running the cell reuses the saved
`spm-a8.model`, so the vocabulary your checkpoint was trained with never changes under it. Pieces
are shorter than sentences, so the cell also measures your longest target in pieces and lets
decoding run that long. These are the settings Assignment 9 adds for everyone else.

In [ ]:
import random

subword_settings = {}
max_decode = {}
if NO_SPACES:
    import pandas as pd
    from torchlingo.data_processing.vocab import SentencePieceVocab
    from torchlingo.preprocessing import train_sentencepiece

    SPM = OUT_DIR / "spm-a8.model"
    if not SPM.exists():
        # character_coverage below 1.0 lets the rarest characters of a large script fall back to
        # <unk> rather than crowd ordinary pieces out of the vocabulary.
        train_sentencepiece([OUT_DIR / "train.tsv"], model_prefix=str(OUT_DIR / "spm-a8"),
                            vocab_size=8_000, character_coverage=0.9995)
    pieces = SentencePieceVocab(str(SPM))
    train_tgt = pd.read_csv(OUT_DIR / "train.tsv", sep="\t", dtype=str, keep_default_na=False).tgt
    longest = int(train_tgt.map(lambda s: len(pieces.encode(s, add_special_tokens=False))).max())
    max_decode = {"max_decode_length": longest + 10}
    subword_settings = {"use_sentencepiece": True,
                        "sp_model_path": str(SPM), "sp_tgt_model_path": str(SPM)}
    print(f"Subword vocabulary: {SPM.name}. Longest training target: {longest} pieces.")

config = Config(
    d_model=512,
    n_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    d_ff=2048,
    batch_size=64,
    num_workers=0,
    **max_decode,
)

# The bucketing shuffles with Python's random, not torch's, so seed it too. Each bucket's last,
# incomplete batch is dropped every epoch: a few hundred sentences, different ones each time.
random.seed(SEED)
train_loader, val_loader, src_vocab, tgt_vocab = create_dataloaders(
    OUT_DIR / "train.tsv",
    OUT_DIR / "val.tsv",
    use_bucketing=True,
    num_workers=0,
    config=config,
    **subword_settings,
)
print(padding_report(train_loader))   # what bucketing saves on YOUR data

torch.manual_seed(SEED)
model = SimpleTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    config=config,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Source vocab {len(src_vocab):,}   target vocab {len(tgt_vocab):,}")
print(f"Model: {n_params:,} parameters")
print(f"Batches per epoch: {len(train_loader):,}   (x {EPOCHS} epochs)")

# Memory was measured with vocabularies up to 50,000 words. A larger one needs more, mostly in
# the output layer; if Step 7 then stops with "CUDA out of memory", this is the setting to change.
if max(len(src_vocab), len(tgt_vocab)) > 50_000:
    print("\nYour vocabulary is larger than the 50,000 words memory was measured with. If Step 7 "
          "runs out of memory, set batch_size=32 in the Config above and run this cell again.")

## The number to compare your first loss against

Lecture 7 derived it: a model that has learned nothing spreads probability evenly, so its
cross-entropy is exactly `ln(V)` for a target vocabulary of size `V`. That is the reference
point. Near it, nothing is learning. Falling below it, something is.

Two things to expect, so neither looks like a failure. The first `log_every` line is the mean of
the first fifty batches, and it usually sits *a little above* `ln(V)`: a freshly built model is
not perfectly uniform, and leaning the wrong way costs more than not leaning at all. And
`train_losses[0]` is the mean of the whole first epoch, by the end of which the model is well
below `ln(V)`. Compare the first logged line with the number here.

In [ ]:
V = len(tgt_vocab)
print(f"ln(V) for V = {V:,} is {uniform_loss(V):.3f}")
print("Your first logged loss should be near this, often a little above. Your epoch-1 average will be below it.")

## Step 7: start training. This continues after class.

Three things this cell does that the handout requires:

- **`val_loader` and `save_dir`**, so the trainer keeps the checkpoint with the lowest
  validation loss, which is not always the last one.
- **A `TrainingCheckpointer`**, which saves resumable state to Drive every ten minutes. If the
  session drops, run this cell again and it picks up where it stopped.
- **`log_every`**, so you see the start of training rather than one number per epoch.

Expect this to take hours, depending on your corpus and your GPU. Start it now, and leave it
running when class ends.

In [ ]:
checkpointer = TrainingCheckpointer(experiment_name="assignment-8")   # Drive, in Colab
BEST_DIR = OUT_DIR / "best"

result = train_model(
    model,
    train_loader,
    val_loader=val_loader,
    num_epochs=EPOCHS,
    device=device,
    config=config,
    gradient_clip=1.0,
    save_dir=BEST_DIR,
    log_every=50,
    checkpointer=checkpointer,
)

print(f"Epochs run: {len(result.train_losses)}")
print(f"Final train loss {result.train_losses[-1]:.4f}   final val loss {result.val_losses[-1]:.4f}")
print(f"Best checkpoint: {result.best_checkpoint}")

## Before you leave the room

- The cell above is running and has printed at least one `log_every` line.
- `OUT_DIR` in Drive has nine files: three `.tsv`, three `.src`, three `.tgt` (and, with
  `NO_SPACES = True`, `spm-a8.model` and `spm-a8.vocab`).
- You wrote down your first logged loss and `ln(V)` next to each other.

If the session drops later: open this notebook, run everything down to and including Step 6
(it is fast; nothing retrains), then run Step 7 again. The checkpointer resumes.

---

## After it finishes: translate the test set and score it

Not for today. Come back to this when Step 7 has completed. Greedy decoding, the whole test set
at once, and SacreBLEU over all 2,000 sentences, the same rule as Assignment 6. Say in your
write-up that you decoded greedily; Lecture 10 explains why that sentence matters.

**With `NO_SPACES = True`, BLEU is computed differently, and your write-up must say so.** BLEU
counts matching words, and SacreBLEU's default finds words at the spaces; on your language it
would score every sentence as one word and report about zero however good the translation.
The cell uses `tokenize="flores200"` instead, often written **spBLEU**: it splits both sides into
pieces with one fixed multilingual SentencePiece model first. It is the BLEU that multilingual
benchmarks such as FLORES and NLLB report. It downloads that model the first time.

In [ ]:
import sacrebleu
from torchlingo.checkpoint import load_checkpoint

# The best-by-validation model, not the last one. It is read from Drive rather than from
# `result`, so this cell also works in a fresh session a day later: run everything down to and
# including Step 6 first (that rebuilds the vocabularies and the model), skip Step 7, then run
# this. load_checkpoint returns a dict; the model is restored from its state dict.
ckpt = load_checkpoint(OUT_DIR / "best" / "model_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])

model.eval()
sources    = list(test.src)
references = list(test.tgt)

hypotheses = []
for start in range(0, len(sources), 64):
    hypotheses.extend(translate_batch(
        model, sources[start:start + 64],
        src_vocab, tgt_vocab,
        decode_strategy="greedy", device=device, config=config,
    ))

# One stream of hypotheses, one list of reference streams. Per-sentence nesting is the
# silent mistake Lecture 6 warned about. Without spaces, spBLEU (see above); otherwise the default.
TOKENIZE = "flores200" if NO_SPACES else "13a"
bleu = sacrebleu.corpus_bleu(hypotheses, [references], tokenize=TOKENIZE)
print(bleu)

(OUT_DIR / "test.hyp").write_text("\n".join(hypotheses) + "\n", encoding="utf-8")
for s, r, h in list(zip(sources, references, hypotheses))[:5]:
    print(f"\nSRC {s}\nREF {r}\nHYP {h}")